# Slippage Model Demonstration

This notebook demonstrates how to fit a slippage model using synthetic data and scikit-learn.
We'll generate synthetic (order_size, slippage) pairs and fit both linear and quantile regressors.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, QuantileRegressor
import joblib

# Add the parent directory to the path so we can import the modules
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), '..'))
from trade_simulator.src.orderbook import OrderBook
from trade_simulator.models.slippage import simulate_slippage, calculate_slippage_percentage

# Set random seed for reproducibility
np.random.seed(42)

## Generate Synthetic Orderbook Data

We'll create a function to generate synthetic orderbook data with varying levels of liquidity.

In [ ]:
def generate_synthetic_orderbook(base_price=50000.0, depth=10, spread=10.0, liquidity_factor=1.0):
    """
    Generate a synthetic orderbook with specified parameters.
    
    Args:
        base_price (float): The base price for the asset
        depth (int): Number of levels to generate on each side
        spread (float): The spread between best bid and best ask
        liquidity_factor (float): Factor to scale the liquidity (higher = more liquidity)
        
    Returns:
        OrderBook: A populated OrderBook instance
    """
    book = OrderBook("BTC-USDT-SWAP")
    
    # Calculate best bid and ask prices
    best_ask = base_price + spread / 2
    best_bid = base_price - spread / 2
    
    # Generate ask levels (ascending prices)
    asks = []
    for i in range(depth):
        price = best_ask + i * 10.0  # Price increases by $10 per level
        size = (1.0 + i * 0.5) * liquidity_factor  # Size increases with price
        asks.append([str(price), str(size)])
    
    # Generate bid levels (descending prices)
    bids = []
    for i in range(depth):
        price = best_bid - i * 10.0  # Price decreases by $10 per level
        size = (1.0 + i * 0.5) * liquidity_factor  # Size increases with depth
        bids.append([str(price), str(size)])
    
    # Create a tick with the generated data
    tick = {
        "timestamp": "2023-01-01T00:00:00Z",
        "exchange": "OKX",
        "symbol": "BTC-USDT-SWAP",
        "asks": asks,
        "bids": bids
    }
    
    book.update_from_tick(tick)
    return book

# Generate a sample orderbook
sample_book = generate_synthetic_orderbook()
print(sample_book)

## Generate Synthetic (Order Size, Slippage) Pairs

Now we'll generate synthetic data points by simulating orders of different sizes and measuring the resulting slippage.

In [ ]:
def generate_slippage_data(n_samples=100, min_order=1000.0, max_order=500000.0, 
                          liquidity_variations=5, side="buy"):
    """
    Generate synthetic (order_size, slippage) pairs.
    
    Args:
        n_samples (int): Number of samples to generate
        min_order (float): Minimum order size in USD
        max_order (float): Maximum order size in USD
        liquidity_variations (int): Number of different liquidity scenarios to generate
        side (str): Order side ("buy" or "sell")
        
    Returns:
        tuple: (order_sizes, slippages) arrays
    """
    # Generate order sizes on a logarithmic scale
    order_sizes = np.exp(np.linspace(np.log(min_order), np.log(max_order), n_samples))
    
    # Add some random noise to order sizes
    order_sizes = order_sizes * np.random.uniform(0.9, 1.1, n_samples)
    
    # Initialize slippage array
    slippages = np.zeros(n_samples)
    slippage_percentages = np.zeros(n_samples)
    
    # Generate data for different liquidity scenarios
    samples_per_variation = n_samples // liquidity_variations
    
    for i in range(liquidity_variations):
        # Vary liquidity factor (higher index = less liquidity)
        liquidity_factor = 2.0 / (i + 1)
        
        # Generate orderbook with this liquidity
        book = generate_synthetic_orderbook(liquidity_factor=liquidity_factor)
        
        # Calculate slippage for this batch of orders
        start_idx = i * samples_per_variation
        end_idx = min((i + 1) * samples_per_variation, n_samples)
        
        for j in range(start_idx, end_idx):
            try:
                # Calculate slippage
                slippages[j] = simulate_slippage(order_sizes[j], book, side)
                slippage_percentages[j] = calculate_slippage_percentage(order_sizes[j], book, side)
            except ValueError:
                # If order is too large for the orderbook, set a high slippage value
                slippages[j] = order_sizes[j] * 0.01  # 1% slippage as a fallback
                slippage_percentages[j] = 1.0
    
    # Add some random noise to slippage to simulate market variability
    slippages = slippages * np.random.uniform(0.8, 1.2, n_samples)
    slippage_percentages = slippage_percentages * np.random.uniform(0.8, 1.2, n_samples)
    
    return order_sizes, slippages, slippage_percentages

# Generate data for buy orders
buy_order_sizes, buy_slippages, buy_slippage_pcts = generate_slippage_data(side="buy")

# Generate data for sell orders
sell_order_sizes, sell_slippages, sell_slippage_pcts = generate_slippage_data(side="sell")

# Create DataFrames for easier manipulation
buy_df = pd.DataFrame({
    'order_size': buy_order_sizes,
    'slippage': buy_slippages,
    'slippage_pct': buy_slippage_pcts,
    'side': 'buy'
})

sell_df = pd.DataFrame({
    'order_size': sell_order_sizes,
    'slippage': sell_slippages,
    'slippage_pct': sell_slippage_pcts,
    'side': 'sell'
})

# Combine the data
df = pd.concat([buy_df, sell_df])

# Display the first few rows
df.head()

## Visualize the Data

Let's plot the relationship between order size and slippage.

In [ ]:
plt.figure(figsize=(12, 6))

# Plot buy orders
plt.subplot(1, 2, 1)
plt.scatter(buy_order_sizes, buy_slippage_pcts, alpha=0.6, label='Buy Orders')
plt.xscale('log')
plt.xlabel('Order Size (USD)')
plt.ylabel('Slippage (%)')
plt.title('Buy Order Slippage')
plt.grid(True, alpha=0.3)

# Plot sell orders
plt.subplot(1, 2, 2)
plt.scatter(sell_order_sizes, sell_slippage_pcts, alpha=0.6, color='orange', label='Sell Orders')
plt.xscale('log')
plt.xlabel('Order Size (USD)')
plt.ylabel('Slippage (%)')
plt.title('Sell Order Slippage')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Fit Linear Regression Model

Now we'll fit a linear regression model to predict slippage based on order size.

In [ ]:
# Prepare features: we'll use log(order_size) as the feature
X_buy = np.log(buy_order_sizes).reshape(-1, 1)
y_buy = buy_slippage_pcts

X_sell = np.log(sell_order_sizes).reshape(-1, 1)
y_sell = sell_slippage_pcts

# Fit linear regression models
buy_linear_model = LinearRegression()
buy_linear_model.fit(X_buy, y_buy)

sell_linear_model = LinearRegression()
sell_linear_model.fit(X_sell, y_sell)

# Make predictions
buy_pred = buy_linear_model.predict(X_buy)
sell_pred = sell_linear_model.predict(X_sell)

# Print model coefficients
print(f"Buy model: slippage_pct = {buy_linear_model.coef_[0]:.6f} * log(order_size) + {buy_linear_model.intercept_:.6f}")
print(f"Sell model: slippage_pct = {sell_linear_model.coef_[0]:.6f} * log(order_size) + {sell_linear_model.intercept_:.6f}")

## Fit Quantile Regression Models

Now we'll fit quantile regression models to capture different percentiles of slippage.

In [ ]:
# Define quantiles to model
quantiles = [0.5, 0.75, 0.9, 0.95]

# Fit quantile regression models for buy orders
buy_quantile_models = {}
for q in quantiles:
    model = QuantileRegressor(quantile=q, alpha=0.5, solver='highs')
    model.fit(X_buy, y_buy)
    buy_quantile_models[q] = model
    print(f"Buy {q*100}% quantile model: slippage_pct = {model.coef_[0]:.6f} * log(order_size) + {model.intercept_:.6f}")

# Fit quantile regression models for sell orders
sell_quantile_models = {}
for q in quantiles:
    model = QuantileRegressor(quantile=q, alpha=0.5, solver='highs')
    model.fit(X_sell, y_sell)
    sell_quantile_models[q] = model
    print(f"Sell {q*100}% quantile model: slippage_pct = {model.coef_[0]:.6f} * log(order_size) + {model.intercept_:.6f}")

## Visualize Model Predictions

Let's plot the data points along with the model predictions.

In [ ]:
# Create a range of order sizes for prediction
order_sizes_range = np.exp(np.linspace(np.log(1000), np.log(500000), 100))
X_range = np.log(order_sizes_range).reshape(-1, 1)

# Predict with linear models
buy_linear_pred = buy_linear_model.predict(X_range)
sell_linear_pred = sell_linear_model.predict(X_range)

# Predict with quantile models
buy_quantile_preds = {q: model.predict(X_range) for q, model in buy_quantile_models.items()}
sell_quantile_preds = {q: model.predict(X_range) for q, model in sell_quantile_models.items()}

# Plot buy orders
plt.figure(figsize=(12, 10))

plt.subplot(2, 1, 1)
plt.scatter(buy_order_sizes, buy_slippage_pcts, alpha=0.6, label='Data Points')
plt.plot(order_sizes_range, buy_linear_pred, 'r-', linewidth=2, label='Linear Regression')

for q, preds in buy_quantile_preds.items():
    plt.plot(order_sizes_range, preds, '--', linewidth=1.5, label=f'{q*100}% Quantile')

plt.xscale('log')
plt.xlabel('Order Size (USD)')
plt.ylabel('Slippage (%)')
plt.title('Buy Order Slippage Models')
plt.grid(True, alpha=0.3)
plt.legend()

# Plot sell orders
plt.subplot(2, 1, 2)
plt.scatter(sell_order_sizes, sell_slippage_pcts, alpha=0.6, color='orange', label='Data Points')
plt.plot(order_sizes_range, sell_linear_pred, 'r-', linewidth=2, label='Linear Regression')

for q, preds in sell_quantile_preds.items():
    plt.plot(order_sizes_range, preds, '--', linewidth=1.5, label=f'{q*100}% Quantile')

plt.xscale('log')
plt.xlabel('Order Size (USD)')
plt.ylabel('Slippage (%)')
plt.title('Sell Order Slippage Models')
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

## Save Models

Finally, let's save the trained models for later use.

In [ ]:
# Create a dictionary with all models
models = {
    'buy_linear': buy_linear_model,
    'sell_linear': sell_linear_model,
    'buy_quantile': buy_quantile_models,
    'sell_quantile': sell_quantile_models
}

# Save models to file
models_dir = os.path.join(os.path.dirname(os.getcwd()), 'models')
os.makedirs(models_dir, exist_ok=True)
model_path = os.path.join(models_dir, 'slippage_model.joblib')
joblib.dump(models, model_path)

print(f"Models saved to {model_path}")

## Model Usage Example

Here's how to load and use the saved models.

In [ ]:
# Load the models
loaded_models = joblib.load(model_path)

# Example: Predict slippage for a $50,000 buy order
order_size = 50000.0
log_order_size = np.log(order_size).reshape(1, -1)

# Predict with linear model
linear_slippage = loaded_models['buy_linear'].predict(log_order_size)[0]
print(f"Linear model prediction for ${order_size} buy order: {linear_slippage:.4f}% slippage")

# Predict with quantile models
for q, model in loaded_models['buy_quantile'].items():
    quantile_slippage = model.predict(log_order_size)[0]
    print(f"{q*100}% quantile prediction for ${order_size} buy order: {quantile_slippage:.4f}% slippage")